In [ ]:
# Python script for the logistic regression case study

notebook_code = """
# Lead Scoring using Logistic Regression for X Education
# Author: Shweta C. Dharmadhikari
# Date: 03/05/2025

# ===========================================
# Step 1: Import Libraries
# ===========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from sklearn.feature_selection import RFE
import statsmodels.api as sm

# ===========================================
# Step 2: Load and Inspect Dataset
# ===========================================
df = pd.read_csv("Leads.csv")
data = df.copy()

# Drop IDs not useful for modeling
data.drop(['Prospect ID', 'Lead Number'], axis=1, inplace=True)

# Replace 'Select' with NaN
data = data.replace('Select', np.nan)

# Drop columns with too many missing values
high_null_cols = ['Lead Quality', 'Asymmetrique Profile Score', 'Asymmetrique Activity Score',
                  'Asymmetrique Profile Index', 'Asymmetrique Activity Index']
data.drop(columns=high_null_cols, inplace=True)

# Drop rows with missing target
data = data[~data['Converted'].isnull()]

# Drop columns with >30% missing and fill small % nulls
data.drop(columns=['Tags', 'What matters most to you in choosing a course',
                   'Lead Profile'], inplace=True)

# Fill mode for categorical and median for numeric
data['City'].fillna(data['City'].mode()[0], inplace=True)
data['Specialization'].fillna(data['Specialization'].mode()[0], inplace=True)
data['TotalVisits'].fillna(data['TotalVisits'].median(), inplace=True)
data['Page Views Per Visit'].fillna(data['Page Views Per Visit'].median(), inplace=True)

# Drop remaining rows with missing values
data.dropna(inplace=True)

# ===========================================
# Step 3: Encode Categorical Variables
# ===========================================
# Get dummies for categorical columns
data_dummies = pd.get_dummies(data, drop_first=True)

# ===========================================
# Step 4: Split Data
# ===========================================
X = data_dummies.drop('Converted', axis=1)
y = data_dummies['Converted']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# ===========================================
# Step 5: Feature Scaling
# ===========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ===========================================
# Step 6: Logistic Regression with RFE
# ===========================================
logreg = LogisticRegression(max_iter=1000)
rfe = RFE(logreg, 20)  # select top 20 features
rfe = rfe.fit(X_train_scaled, y_train)

# Select columns based on RFE
selected_cols = X_train.columns[rfe.support_]
X_train_rfe = X_train[selected_cols]
X_test_rfe = X_test[selected_cols]

# Scale selected columns
X_train_rfe_scaled = scaler.fit_transform(X_train_rfe)
X_test_rfe_scaled = scaler.transform(X_test_rfe)

# ===========================================
# Step 7: Fit Final Model
# ===========================================
model = LogisticRegression()
model.fit(X_train_rfe_scaled, y_train)

# ===========================================
# Step 8: Model Evaluation
# ===========================================
y_pred = model.predict(X_test_rfe_scaled)
y_prob = model.predict_proba(X_test_rfe_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

# Confusion Matrix
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, label="ROC Curve (area = %0.2f)" % roc_auc_score(y_test, y_prob))
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()

# ===========================================
# Step 9: Lead Scoring
# ===========================================
lead_scores = pd.DataFrame()
lead_scores["Lead Score"] = y_prob * 100
lead_scores["Predicted"] = y_pred
lead_scores["Actual"] = y_test.values
lead_scores.head()

# You can save the lead scores to a CSV if needed
# lead_scores.to_csv("LeadScores.csv", index=False)
"""

# Save as a .py file
script_path = "/mnt/data/Lead_Scoring_Logistic_Regression.py"
with open(script_path, "w") as f:
    f.write(notebook_code)

script_path
